[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/electrokinetics/electrokinetics.ipynb)

# Electrokinetic Transport

This example computes the flow and the electric current that a pressure drop or an electric field drives through channels between charged walls whose width and wall charge vary along them, on a mesh of triangles that covers each channel whole. A charged wall draws ions of the opposite charge into the liquid beside it, a thin cloud called the electric double layer. A pressure drop pushes the liquid and carries the cloud with it, a current called the streaming current; a field pushes the cloud, and the cloud drags the liquid along, a flow called electro-osmosis; and every ion also drifts through the liquid under the field, which is conduction. Onsager's reciprocal relation makes the two cross effects one number. Where the walls change along a channel, the liquid finds its own way between them and is stirred into cells, and a wavy channel whose walls carry no net charge still pumps, as [Ghosal (2002)](https://doi.org/10.1017/S0022112002007899) shows for channels much longer than they are wide; here each channel is solved whole. Last, the four responses of straight slits bound how much of a pressure drop's power a slit can deliver as electric power, as [van der Heyden, Bonthuis, Stein, Meyer and Dekker (2006)](https://doi.org/10.1021/nl061524l) work out.

The algebra is planar spacetime joined with one more direction, `v`, for the volume a flow carries, `Algebra("x+y+t-") * Algebra("v0")`, degenerate since nothing measures a volume against a length. A parcel of the solution is modelled as a point whose weight is its volume, `mv.v.dual()`, offset along time by the charge it holds, and an ion, charge without volume, as a point at infinity along time. A drive is a field of bivectors, minus the gradient of the potentials each wedged with what it acts on, `-(gradient ^ (pressure * mv.v + potential * mv.t))`: along the channel it holds the applied pressure drop, `mv.xv`, and field, `mv.xt`, and across it, `mv.yv` and `mv.yt`, the double layer's own osmotic pressure and field. A drive's commutator with a parcel is the force on it, `drive.commutator(parcel)`, and a drive reads a flux by the regressive product, `drive & flux`, the power it spends. The flow is a stream function on the vertices, whose rise along an edge is the volume crossing it, so that no volume piles up anywhere; the potential the liquid adds is the one whose every hat's drive reads the channel's flux as nothing, so that no charge piles up either. The reciprocity, `battery & flux(pump) == pump & flux(battery)`, comes out of the computation rather than going into it, and the entropy produced is two regressive pairings, the liquid's turning with the viscous moment that resists it and the field with the current it conducts. [The capillary electrophoresis notebook](electrophoresis.ipynb) follows a band of dye along a straight channel, carried by these flows and drifting by its charge.

In the notation of continuum electrokinetics, the flow solves the Stokes equations with the cloud's body force, $\eta\nabla^2\mathbf u = \nabla p - \rho\mathbf E$ and $\nabla\cdot\mathbf u = 0$, here through the stream function $\psi$, $\mathbf u = (\partial_y\psi, -\partial_x\psi)$, as $\eta\nabla^4\psi = \nabla\times(\rho\mathbf E)$; the current conserves charge, $\nabla\cdot(\sigma\mathbf E + \rho\mathbf u) = 0$; and the double layer solves the Poisson–Boltzmann equation, $\epsilon\nabla^2\phi = -\rho$ with the ions in their Boltzmann distribution. The slit's response is the matrix of the coefficients $L_{ij}$, with $L_{12} = L_{21}$, and its best efficiency as a generator is $\alpha/(1+\sqrt{1-\alpha})^2$, with $\alpha = L_{12}^2/(L_{11}L_{22})$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.sparse import spdiag
from examples.fluids.electrokinetics import render, scenarios
from examples.fluids.electrokinetics.numerical import solve_coupled

# Planar spacetime, and the volume a flow carries: a direction of its own, degenerate, since nothing measures a volume
# against a length or a time.
spacetime = Algebra("x+y+t-")
ga = spacetime * Algebra("v0")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Point = ga.gatype.antivector()                                                # Antivector-
Plane = ga.gatype.vector()                                                    # Vector+
# A drive: the electric potential's gradient wedged with time, and the pressure's, wedged with volume.
Drive = ga.gatype.from_blades("xt yt xv yv")                                  # Bivector+
Flux = ga.gatype.antibivector()                                               # Antibivector-
axis = mv.x.dual()                                                            # [] Point-
across = mv.y.dual()                                                          # [] Point-
time = -mv.t.dual()                                                           # [] Point-
# A unit of volume, the weight of a point.
volume = mv.v.dual()                                                          # [] Point-
# A pump, a unit pressure drop along the channel, and a battery, a unit field along it.
pump, battery = mv.xv, mv.xt                                                  # [] Drive+, Drive+

## The channels

Three channels share one strip of triangles, closed on itself along its length: a chain of vertices across, swept around a loop along, so that the strip has walls but no ends. Its width swings along it in a wave, and the walls of each channel carry a charge in a wave of the same length with no net charge, in step with the width's wave in the first channel, a quarter wave behind it in the second and half a wave behind in the third. The strip and the charge its walls put on each wall vertex come from the scenario; its vertices are places of unit volume, and an edge that crosses from the last column back to the first reaches a period further along than its ends' places. Lengths are in Debye lengths, the distance over which the bulk's ions screen a charge, and a wall's charge per area in units of the permittivity times a thermal volt per Debye length, about 2 mC/m² in millimolar salt.

In [ ]:
strip, strip_places, strip_shifts, strip_lower, strip_upper, strip_charge, wall_charges = scenarios.channels()
# Each vertex a place of unit volume; each edge from its tail to its head, across the seam a period further along.
strip_vertices = (volume + axis * strip_places[..., 0] + across * strip_places[..., 1]).field()   # Point-[vertices]
strip_edges = strip.d[0] * strip_vertices + (axis * strip_shifts).field()                         # Point-[edges]
render.draw_mesh(strip, strip_vertices, scenarios.n_across, scenarios.wavelength, wall_charges[0]);

The strip's triangles, between walls coloured by the first channel's charge, magenta where positive and green where negative.

## The double layer

Each ion settles where its energy in the potential, its valence times the potential, balances its spreading: its concentration is the bulk's times the exponential of minus that energy, the Boltzmann distribution. The potential is in turn set by the charge: the field's outflow from each vertex's cell equals the cell's charge, Gauss's law, with the walls' own charge in the cells along them. On the triangles, each corner's hat rises from nothing on the facing side to one at the corner, and its gradient is the plane through that side, along time and at unit volume, over twice the triangle's area, `(time & volume & sides) / double_areas`, with no metric. The metric enters with the material: a potential rising by one at a corner drives the field down its hat's gradient, `pushes = -(gradients ^ mv.t)`, the vacuum answers each drive with its excitation, its push on a unit charge swept through time, `pushes.commutator(time).regressive(time)`, the permittivity being one, and each hat's drive reading every other's excitation, gathered over the triangles, is the field's outflow. A damped Newton's method solves the two together from the bulk's potential. Potentials are in thermal volts, the thermal energy per unit charge.

In the notation of finite elements, the field's outflow is the stiffness matrix $K_{ij} = \int \nabla\phi_i\cdot\nabla\phi_j\,\mathrm{d}A$ of the hat functions $\phi_i$, the metric's pairing of two gradients.

In [ ]:
# Potassium chloride, its bulk concentrations giving a Debye length of one, `(valences ** 2 * bulk).sum() == 1`; the
# Newton steps that settle a double layer.
valences = np.array([1, -1])
bulk = np.array([0.5, 0.5])
n_newton_steps = 20

In [ ]:
def triangles(topology, edges: Point) -> tuple[Point, Plane, Scalar]:
    """The side facing each corner of each triangle `[..., corners] Point-[faces]`, each corner's hat gradient
    `[..., corners] Plane+[faces]`, and each triangle's area `[...] Scalar+[faces]`."""
    sides = topology.boundary[2, 1] * edges[..., None]                                  # [..., corners] Point-[faces]
    # Twice a triangle's area: two of its sides joined, read through time at unit volume.
    double_areas = ((sides[..., 0] & sides[..., 1]) & time) & volume                   # [...] Scalar-[faces]
    gradients = (time & volume & sides) / double_areas[..., None]                      # [..., corners] Plane+[faces]
    # The integrals weight by each area's size, an ordinary number.
    return sides, gradients, double_areas.norm() / 2


def gathered(topology, local: Scalar):
    """A form on each triangle's corners `[..., corners, corners] Scalar+[faces]`, gathered onto the vertices it joins."""
    corners = topology.incidence[2, 0]                                                  # [corners] Scalar+[faces <- vertices]
    return (~corners[:, None] * spdiag(local) * corners[None, :]).sum(axis=-1).sum(axis=-1)


def double_layer(topology, edges: Point, wall_charge: Scalar) -> Scalar:
    """The potential over the strip `[...] Scalar+[vertices]`, for the charge its walls put on its vertices."""
    _, gradients, areas = triangles(topology, edges)
    # Each hat's drive, the excitation the vacuum answers it with, and the field's outflow from each vertex's cell, each
    # drive reading every other's excitation; and the cells' areas, a third of each triangle around.
    pushes = -(gradients ^ mv.t)                                                        # [..., corners] Drive+[faces]
    excitations = pushes.commutator(time).regressive(time)                              # [..., corners] Flux-[faces]
    outflow = gathered(topology, (pushes[..., :, None] & excitations[..., None, :]) * areas[..., None, None])
    cells = ~topology.incidence[2, 0].sum(axis=0) * (areas / 3)                         # [...] Scalar+[vertices]
    potential = 0 * wall_charge                                                         # [...] Scalar+[vertices]
    for _ in range(n_newton_steps):
        concentrations = bulk * (-valences * potential[..., None]).exp()                # [..., species] Scalar+[vertices]
        residual = outflow * potential - cells * (valences * concentrations).sum(axis=-1) - wall_charge
        # The charge a cell loses as its potential rises, by its ions' squared valences.
        susceptibility = (valences ** 2 * concentrations).sum(axis=-1)                  # [...] Scalar+[vertices]
        step = ((outflow + spdiag(cells * susceptibility)) * Scalar).solve(residual)    # [...] Scalar+[vertices]
        # Newton's step, held below a thermal volt so that no Boltzmann factor changes by much more than e at once.
        potential = potential - step.tanh()
    return potential


strip_potentials = double_layer(strip, strip_edges, mv.scalar(strip_charge[..., None]).field())   # [channels] Scalar+[vertices]
densities = (valences * bulk * (-valences * strip_potentials[..., None]).exp()).sum(axis=-1)       # [channels] Scalar+[vertices]
render.draw_charge(strip, strip_vertices, scenarios.n_across, scenarios.wavelength, densities, wall_charges);

The liquid's charge colours each channel, magenta where positive and green where negative, on one scale for all three: each stretch of wall gathers a cloud of the opposite charge, and where the wall's charge changes sign the cloud does too, across the whole width where the channel narrows.

Across the channel the cloud is in equilibrium. Its ions' excess over the bulk's is an osmotic pressure, each ion pushing as a molecule of a gas does, and its drive is that pressure's gradient wedged with volume, `-(gradient ^ (osmotic * mv.v))`; the cloud's own field is its potential's gradient wedged with time, `-(gradient ^ (potential * mv.t))`. Both lie across the channel, `mv.yv` and `mv.yt`, the transverse blades of the joined algebra, and each pushes the parcels by its commutator with them, the one on their volume and the other on their charge.

In [ ]:
strip_sides, strip_gradients, areas = triangles(strip, strip_edges)
corners = strip.incidence[2, 0]                                                         # [corners] Scalar+[faces <- vertices]
strip_cells = ~corners.sum(axis=0) * (areas / 3)                                        # Scalar+[vertices]
strip_concentrations = corners.mean(axis=0) * (bulk * (-valences * strip_potentials[..., None]).exp())   # [channels, species] Scalar+[faces]
strip_parcels = volume + (time * valences * strip_concentrations).sum(axis=-1)         # [channels] Point-[faces]
osmotic = (bulk * ((-valences * strip_potentials[..., None]).exp() - 1)).sum(axis=-1)  # [channels] Scalar+[vertices]
osmotic_drive = -(((corners * osmotic[..., None]) * strip_gradients).sum(axis=-1) ^ mv.v)            # [channels] Bivector+[faces]
electric_drive = -(((corners * strip_potentials[..., None]) * strip_gradients).sum(axis=-1) ^ mv.t)  # [channels] Bivector+[faces]
# Each drive's push on the parcels, and at each vertex its triangles' weighed by their areas.
osmotic_push = (~corners.sum(axis=0) * (osmotic_drive.commutator(strip_parcels) * (areas / 3))) / strip_cells     # [channels] Point-[vertices]
electric_push = (~corners.sum(axis=0) * (electric_drive.commutator(strip_parcels) * (areas / 3))) / strip_cells   # [channels] Point-[vertices]
render.draw_balance(strip, strip_vertices, scenarios.n_across, scenarios.wavelength, osmotic_push[0], electric_push[0],
                    wall_charges[0]);

The first channel three times, on one scale, a square root so that the double layers' reach shows beside their peak at the walls: coloured by the push across it of its double layer's osmotic pressure, of its own field, and of the two together, orange where the push is up the picture and purple where down. Along each wall the ions crowding there push the liquid away from it, as a gas presses outward, and the wall's charge pulls theirs back towards it just as hard; where the wall's charge changes sign there is no cloud and no push. Together the two push nothing, up to how finely the triangles resolve the cloud's steep edge. A pressure and a field are two kinds of one thing here, a potential's gradient wedged with what the potential acts on, and in the double layer they hold each other in balance.

## Forces, flows and fluxes

A drive's commutator with a parcel is the force on it: the pressure drop pushes every unit of volume alike, `pump.commutator(volume) == axis`, and passes over the ions, which have none; the field pushes every unit of charge alike, `battery.commutator(time) == axis`, and passes over the volume. In the usual formulation the force on the liquid, the pressure gradient plus the charge density times the field, is written down term by term, and that a pressure drop does not push the ions through the liquid is a choice made in the model; here both follow from the one commutator and the types of what it acts on.

The liquid's flow is a stream function, rising along an edge by the volume that crosses it, so that what flows into a cell flows out of it. A stream function rising by one at a corner carries the liquid along the facing side, half the side's worth over the triangle, `sides / 2`, and viscosity resists a flow by its vorticity, its circulation around each vertex's cell over the cell's area. The circulation is where the metric enters the flow: a velocity read as the plane that takes its circulation along a path, `circulating(velocity) & path`, and over a triangle each corner's velocity read so by every other's is its circulation around the other's cell. The walls hold the stream function to one value along each, the volume flowing between them, and leave the vorticity there free. A potential rising by one at a corner drives the field down its hat's gradient, `-(gradients ^ mv.t)`. The drives do work on a flow by reading what it carries, and the potential the liquid adds is the one every hat's drive reads the channel's flux as nothing: so the force balance and the current's conservation are one system, which leaves the algebra for SciPy's sparse solve in one helper. An ion of valence z conducts as z squared times the flux of a unit charge drifting under a drive, `drive.commutator(time).regressive(time)`. The ions stay in their Boltzmann distribution, in local equilibrium with the potential of the double layer.

In [ ]:
viscosity = 1.0
# The mobilities through water of potassium and chloride ions, in units of the permittivity times a thermal volt over
# the viscosity.
mobilities = np.array([4.27, 4.43])

In [ ]:
# A velocity read as the plane that takes its circulation along a path: its run along and across, read back.
circulating = (stack((mv.x, mv.y)) * (stack((mv.x, mv.y)) & Point)).sum(axis=0)    # [] Plane+ <- Point-


def transport(topology, edges: Point, potential: Scalar, periods: np.ndarray, lower: np.ndarray, upper: np.ndarray,
              drives: Drive) -> tuple[Scalar, Scalar, Flux]:
    """For each drive applied along the strip `[...]`, broadcast against the strips: the stream function and the
    potential the liquid adds `[...] Scalar+[vertices]`, and what the strip carries along per length `[...] Flux-`."""
    corners = topology.incidence[2, 0]                                                  # [corners] Scalar+[faces <- vertices]
    sides, gradients, areas = triangles(topology, edges)
    cells = ~corners.sum(axis=0) * (areas / 3)                                          # [...] Scalar+[vertices]
    # The ions in each triangle, the mean of its corners'; a parcel is its volume offset along time by their charge.
    concentrations = corners.mean(axis=0) * (bulk * (-valences * potential[..., None]).exp())   # [..., species] Scalar+[faces]
    parcels = volume + (time * valences * concentrations).sum(axis=-1)                 # [...] Point-[faces]
    conductivity = (mobilities * valences ** 2 * concentrations).sum(axis=-1)          # [...] Scalar+[faces]
    # What a stream function rising by one at a corner carries over the triangle, and how a potential rising by one
    # there drives, and what its ions conduct.
    flows = sides / 2                                                                   # [..., corners] Point-[faces]
    pushes = -(gradients ^ mv.t)                                                        # [..., corners] Bivector+[faces]
    conducts = pushes.commutator(time).regressive(time)                                 # [..., corners] Flux-[faces]
    # The velocity of each corner's flow, read as the plane of its circulation by every other's over the triangle: its
    # circulation around the other's cell, and over the cell's area, its vorticity, which viscosity resists.
    hat_velocities = flows / areas[..., None]                                           # [..., corners] Point-[faces]
    circulation = gathered(topology, (circulating(hat_velocities[..., :, None]) & hat_velocities[..., None, :])
                           * areas[..., None, None])
    viscous = viscosity * circulation * spdiag(1 / cells) * circulation                 # Scalar+[..., vertices <- vertices]
    # A potential's drive reads what a flow carries, and what the ions conduct under another potential's drive.
    carried = flows & parcels[..., None]                                                # [..., corners] Flux-[faces]
    coupling = gathered(topology, carried[..., :, None] & pushes[..., None, :])
    conduction = gathered(topology, (pushes[..., :, None] & conducts[..., None, :]) * (conductivity * areas)[..., None, None])
    # The applied drive's work on each flow, and its ions' current read by each potential's drive.
    flow_load = (~corners * (drives[..., None] & carried)).sum(axis=-1)                # [...] Scalar+[vertices]
    applied_current = conductivity * drives.commutator(time).regressive(time)          # [...] Flux-[faces]
    current_load = (~corners * ((pushes & applied_current[..., None]) * areas[..., None])).sum(axis=-1)
    stream, added = solve_coupled(viscous, coupling, conduction, flow_load, current_load, topology.interior[0],
                                  upper, int(lower[0]))
    # The strip's flux per length: the parcels with the flow, the ions with their drift under the whole drive.
    velocities = ((corners * stream[..., None]) * sides).sum(axis=-1) / (2 * areas)    # [...] Point-[faces]
    field = drives + ((corners * added[..., None]) * pushes).sum(axis=-1)              # [...] Drive+[faces]
    flux = velocities.regressive(parcels) + conductivity * field.commutator(time).regressive(time)   # [...] Flux-[faces]
    return stream, added, (flux * areas).sites.sum() / periods

## Reciprocity

Two experiments probe each channel: the pump and the battery. Each drive reads each experiment's flux, the pump the volume the flux carries and the battery its charge. A drive's reading of its own flux is the power it spends, the temperature times the entropy it produces, and every drive spends some. The battery's reading of the pump's flux is the streaming current, the pump's reading of the battery's flux the electro-osmotic flow, and the two are equal, for each channel as a whole.

Van der Heyden et al. write the Onsager relation into their model and compute the streaming current alone; here both cross effects are computed, from two experiments, and the table comes out symmetric. The reason is in the structure: the coupling that turns a potential's drive into work on a flow is the same that turns the flow's convected charge into current read by that drive, `carried & pushes` gathered once, and the viscous and the conducting maps are symmetric forms.

In [ ]:
experiments = stack((pump, battery))                                                    # [experiments] Drive+
streams, added, fluxes = transport(strip, strip_edges, strip_potentials, np.array(scenarios.wavelength), strip_lower,
                                   strip_upper, experiments[:, None])                  # [experiments, channels] ...
# Each drive's reading of each experiment's flux, through each channel.
readings = experiments[:, None, None] & fluxes[None]                                    # [drives, experiments, channels] Scalar+

In [ ]:
np.set_printoptions(precision=4)
print("each channel's readings, the pump's and the battery's, of the pump's flux and of the battery's:")
print(np.moveaxis(readings.to_array(), -1, 0))

## The flow

Under the battery alone, the liquid's velocity in each triangle is the stream function's rise around it, and at each vertex its triangles' velocities weighed by their areas.

In [ ]:
stream = streams[1]                                                                     # [channels] Scalar+[vertices]
velocities = ((corners * stream[..., None]) * strip_sides).sum(axis=-1) / (2 * areas)  # [channels] Point-[faces]
speeds = (~corners.sum(axis=0) * (velocities * (areas / 3))) / strip_cells             # [channels] Point-[vertices]
render.draw_flow(strip, strip_vertices, scenarios.n_across, scenarios.wavelength, speeds, wall_charges);

Red is liquid running with the field and blue against it, and the dark streaks run along the liquid's velocity, each as long as the liquid there is fast and fading the way it goes, on one scale for all three channels. With the charge's wave in step with the width's, the channel pumps along the field; half a wave behind, it pumps against the field, though every wall carries as much positive charge as negative; and a quarter wave behind, it all but stops pumping and only turns the liquid in closed cells. In the two that pump, the clouds along the walls of the wide stretches drag the liquid beside them against the stream, and it turns there in small cells of its own. Chloride ions are a little more mobile than potassium ions, so the walls' negative stretches, screened by potassium, and their positive ones, screened by chloride, are not each other's mirror images, and the channel a quarter wave behind still pumps a little.

## Where the entropy is produced

The power the battery spends on a channel, `battery & flux`, is the temperature times the entropy produced in it, and the liquid makes it into heat in two ways. Its turning, its circulation around each vertex's cell over the cell's area, is a rate in the plane of the flow, `mv.xy * vorticity`, and viscosity resists it with a moment in that plane's patch, `resisting = patch * (Rotation & patch)` times the viscosity, an intrinsic patch from an extrinsic rate; the heat is the rate read by the moment, `rotation & (viscosity * resisting(rotation))`. The ions' drift is driven by the whole field, the battery's and the potential the liquid adds, and the heat is the field read by the current it conducts, `field & current`. Added over a channel, the two make exactly the power the battery spends.

In [ ]:
# The patch the plane of the flow spans, and a turning in it read back as the moment that resists it, per unit of
# viscosity.
patch = axis & across                                                                   # [] Patch-
Rotation = ga.gatype.from_blades("xy")                                                  # Bivector+
resisting = patch * (Rotation & patch)                                                  # [] Patch- <- Bivector+
# The liquid's turning: its circulation around each vertex's cell, over the cell's area.
hat_velocities = strip_sides / (2 * areas)                                              # [corners] Point-[faces]
circulation = gathered(strip, (circulating(hat_velocities[:, None]) & hat_velocities[None, :]) * areas)
rotation = mv.xy * ((circulation * stream) / strip_cells)                               # [channels] Bivector+[vertices]
viscous_heat = rotation & (viscosity * resisting(rotation))                             # [channels] Scalar+[vertices]
# The whole field in each triangle, the battery's and the one the liquid adds, and the current its ions conduct.
pushes = -(strip_gradients ^ mv.t)                                                      # [corners] Bivector+[faces]
field = battery + ((corners * added[1][..., None]) * pushes).sum(axis=-1)            # [channels] Drive+[faces]
current = (mobilities * valences ** 2 * strip_concentrations).sum(axis=-1) * field.commutator(time).regressive(time)   # [channels] Flux-[faces]
joule_heat = (~corners.sum(axis=0) * ((field & current) * areas / 3)) / strip_cells    # [channels] Scalar+[vertices]
heat = viscous_heat + joule_heat                                                        # [channels] Scalar+[vertices]
render.draw_heat(strip, strip_vertices, scenarios.n_across, scenarios.wavelength, heat, wall_charges);

The heat colours each channel from white for none to orange for the most in any of them. All but about a percent of it is the ions' friction, the current's Joule heating, made where the channel narrows, since the same current passes through less liquid there; in the channel whose charge's wave is a quarter wave behind its width's, the walls are uncharged at the narrowest place, so fewer counter-ions carry the current there, and the heat gathers in it.

## A slit as a generator

Last, straight slits. A slit pumped with its ends joined through a load drives a current through the load, against the field that builds up along the slit; its efficiency is the electric power delivered over the pumping power spent. It depends on the four responses only through the coupling, the cross effect squared over the two direct ones, and at the best load it is `coupling / (2 - coupling + 2 * root)`, with `root` the square root of one less the coupling. Slits of many half widths and wall charges make one batch of short strips, each as long as it is wide. This cell takes about five seconds, most of it settling the double layers and solving the strips one by one.

In [ ]:
generators, generator_places, generator_shifts, generator_lower, generator_upper, generator_charge, lengths = scenarios.generators()
generator_vertices = (volume + axis * generator_places[..., 0] + across * generator_places[..., 1]).field()   # [widths, charges] Point-[vertices]
generator_edges = generators.d[0] * generator_vertices + (axis * generator_shifts).field()                     # [widths, charges] Point-[edges]
generator_potentials = double_layer(generators, generator_edges, mv.scalar(generator_charge[..., None]).field())
_, _, generator_fluxes = transport(generators, generator_edges, generator_potentials, lengths, generator_lower,
                                   generator_upper, experiments[:, None, None])                                # [experiments, widths, charges] Flux-
generator_readings = experiments[:, None, None, None] & generator_fluxes[None]          # [drives, experiments, widths, charges] Scalar+
# The coupling: the cross effect squared, over the two direct ones.
coupling = generator_readings[0, 1].squared() / (generator_readings[0, 0] * generator_readings[1, 1])   # [widths, charges] Scalar+
# The square root of one less the coupling, as the exponential of half its logarithm: numga has no square root of a
# scalar.
root = ((1 - coupling).log() / 2).exp()                                                 # [widths, charges] Scalar+
efficiency = coupling / (2 - coupling + 2 * root)                                       # [widths, charges] Scalar+
render.draw_efficiency(scenarios.half_widths, scenarios.surface_charges, efficiency);

The best efficiency is shaded from nothing, white, to its largest, mid blue, with level lines at 1, 2, 4 and 6 percent, over the half width and the size of the walls' charge, both on logarithmic axes. In slits narrower than a Debye length the efficiency depends on the two only through their product, and it peaks, near 6.4 percent, along the ridge where the half width is about eight Gouy–Chapman lengths, `2 / abs(surface_charge)` here, the thickness of the layer that holds most of the counter-ions, as van der Heyden et al. find in their low-salt analysis. Less charged than that, the counter-ions spread across the slit where the flow carries them well, but there are few of them; more charged, they pack into a thin layer at the walls, where the liquid barely moves, and conduct all the more. Wider slits let co-ions in, which carry current back without the flow carrying them, and the efficiency falls.

In [ ]:
# checks
# The pressure drop pushes a unit of volume along the channel and passes over an ion; the field pushes a unit of
# charge and passes over the volume: each push read by the plane x.
pushed = stack((pump.commutator(volume), pump.commutator(time), battery.commutator(time), battery.commutator(volume)))
np.testing.assert_allclose((mv.x & pushed).to_array().reshape(-1), [1, 0, 1, 0])
# Across the channels the double layer is in equilibrium: its osmotic pressure's push and its own field's cancel, up to
# the triangles' resolution of its steep edge.
across_osmotic = (mv.y & osmotic_push).to_array()
across_net = (mv.y & (osmotic_push + electric_push)).to_array()
assert np.abs(across_net).max() < 0.1 * np.abs(across_osmotic).max()
# Each channel's response is reciprocal, and every drive spends power.
tables = np.moveaxis(readings.to_array(), -1, 0)                                        # [channels, drives, experiments]
np.testing.assert_allclose(tables, np.swapaxes(tables, -1, -2), rtol=1e-10)
assert np.all(np.linalg.eigvalsh(tables) > 0)
# The channels pump along the field in step, against it half a wave behind, and all but nothing a quarter wave behind.
pumped = (pump & fluxes[1]).to_array().reshape(-1)
assert pumped[0] > 0 > pumped[2]
assert abs(pumped[1]) < 0.05 * pumped[0]
# All the power the battery spends on each channel is made into heat in its liquid.
made = (heat * strip_cells).sites.sum()                                                 # [channels] Scalar+
spent = (battery & fluxes[1]) * scenarios.wavelength                                    # [channels] Scalar+
np.testing.assert_allclose(made.to_array(), spent.to_array(), rtol=1e-10)
# The best efficiency, near 6.4 percent, where the half width is about eight Gouy-Chapman lengths.
best = np.unravel_index(efficiency.argmax(), efficiency.shape)
np.testing.assert_allclose(efficiency[best].to_array(), 0.064, atol=3e-3)
assert 6 < scenarios.half_widths[best[0]] * np.abs(scenarios.surface_charges[best[1]]) / 2 < 11